# Project 3 Debt & Economic Size

## Part 1: Introduction
**Datasets to be used:**
1. World Bank external debt stocks, long-term (DOD, Current US$) (Multilateral debt) [https://data360.worldbank.org/en/indicator/WB_IDS_DT_DOD_MLAT]
2. World Bank GDP (Current US$) (GDP)[https://data360.worldbank.org/en/indicator/WB_WDI_NY_GDP_MKTP_CD]

**Analysis Question:**
Is long-term external debt positively correlated with GDP across countries, and do developing countries economies exhibit higher Debt-to-Ratios indicating greater burdens over the period 2000-2023?

**Hypothesis:**
Larger economies will have more external debt in absolute terms.Developing countries are expected to have higher Debt-to-GDP ratios, reflecting greater financial vulnerability compared to advanced economies.

**Columns Used:**
- Ref Area Label- Country name
- Ref Area- Country code
- Time Period- Year
- Debt_USD- Long term external debt (US$)
- GDP_USD- Gross domestic product (US$)

**Merging Keys:**
- Ref Area
- Time Period


## Part 2: Organizing and Merging the datasets

### 1. Load the datasets

In [158]:
import pandas as pd

Debt= pd.read_csv("WB_IDS_DT_DOD_MLAT.csv")
GDP= pd.read_csv("WB_WDI_NY_GDP_MKTP_CD.csv")


In [159]:
Debt.head()

,STRUCTURE,STRUCTURE_ID,ACTION,FREQ,FREQ_LABEL,REF_AREA,REF_AREA_LABEL,INDICATOR,INDICATOR_LABEL,SEX,...,UNIT_MULT,UNIT_MULT_LABEL,UNIT_TYPE,UNIT_TYPE_LABEL,TIME_FORMAT,TIME_FORMAT_LABEL,OBS_STATUS,OBS_STATUS_LABEL,OBS_CONF,OBS_CONF_LABEL
0,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_IDS_DT_DOD_MLAT,Multilateral debt,_T,...,0,Units,RATIO,Ratio,602,CCYY,O,Missing value,PU,Public
1,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_IDS_DT_DOD_MLAT,Multilateral debt,_T,...,0,Units,RATIO,Ratio,602,CCYY,O,Missing value,PU,Public
2,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_IDS_DT_DOD_MLAT,Multilateral debt,_T,...,0,Units,RATIO,Ratio,602,CCYY,O,Missing value,PU,Public
3,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_IDS_DT_DOD_MLAT,Multilateral debt,_T,...,0,Units,RATIO,Ratio,602,CCYY,O,Missing value,PU,Public
4,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_IDS_DT_DOD_MLAT,Multilateral debt,_T,...,0,Units,RATIO,Ratio,602,CCYY,O,Missing value,PU,Public


In [160]:
GDP.head()

,STRUCTURE,STRUCTURE_ID,ACTION,FREQ,FREQ_LABEL,REF_AREA,REF_AREA_LABEL,INDICATOR,INDICATOR_LABEL,SEX,...,DATA_SOURCE_LABEL,UNIT_TYPE,UNIT_TYPE_LABEL,TIME_FORMAT,TIME_FORMAT_LABEL,COMMENT_OBS,OBS_STATUS,OBS_STATUS_LABEL,OBS_CONF,OBS_CONF_LABEL
0,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,Annual,AFE,Africa Eastern and Southern,WB_WDI_NY_GDP_MKTP_CD,GDP (current US$),_T,...,World Development Indicators (WDI),CUR,Currency,P1Y,Annual,NaN,A,Normal value,PU,Public
1,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,Annual,AFW,Africa Western and Central,WB_WDI_NY_GDP_MKTP_CD,GDP (current US$),_T,...,World Development Indicators (WDI),CUR,Currency,P1Y,Annual,NaN,A,Normal value,PU,Public
2,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,Annual,CSS,Caribbean small states,WB_WDI_NY_GDP_MKTP_CD,GDP (current US$),_T,...,World Development Indicators (WDI),CUR,Currency,P1Y,Annual,NaN,A,Normal value,PU,Public
3,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,Annual,EAR,Early-demographic dividend,WB_WDI_NY_GDP_MKTP_CD,GDP (current US$),_T,...,World Development Indicators (WDI),CUR,Currency,P1Y,Annual,NaN,A,Normal value,PU,Public
4,datastructure,WB.DATA360:DS_DATA360(1.3),I,A,Annual,EAS,East Asia & Pacific,WB_WDI_NY_GDP_MKTP_CD,GDP (current US$),_T,...,World Development Indicators (WDI),CUR,Currency,P1Y,Annual,NaN,A,Normal value,PU,Public


### 2. Reshape to tidy format

In [161]:
Debt = Debt[["REF_AREA_LABEL", "REF_AREA", "TIME_PERIOD","OBS_VALUE"]].rename(
    columns={"OBS_VALUE": "Debt_USD"}
)

GDP = GDP[["REF_AREA_LABEL", "REF_AREA", "TIME_PERIOD","OBS_VALUE"]].rename(
    columns={"OBS_VALUE": "GDP_USD"}
)

print("Debt rows:", len(Debt))
print("GDP rows:", len(GDP))


Debt rows: 351230
GDP rows: 14541


### 3. Merging the Debt and GDP datasets into one table

In [162]:
df = pd.merge(
    Debt[["REF_AREA", "REF_AREA_LABEL", "TIME_PERIOD", "Debt_USD"]],
    GDP[["REF_AREA", "TIME_PERIOD", "GDP_USD"]],
    on=["REF_AREA", "TIME_PERIOD"],
    how="inner"
)

print("merged rows:", len(df))
print(df.head())


merged rows: 290413
  REF_AREA REF_AREA_LABEL  TIME_PERIOD  Debt_USD       GDP_USD
0      ARG      Argentina         1970       0.0  3.158421e+10
1      ARG      Argentina         1970       0.0  3.158421e+10
2      ARG      Argentina         1970       0.0  3.158421e+10
3      ARG      Argentina         1970       0.0  3.158421e+10
4      ARG      Argentina         1970       0.0  3.158421e+10


In [163]:
df = df.dropna(subset=["Debt_USD", "GDP_USD"])
df = df[df["GDP_USD"] > 0]

### 4. Calculate Debt-to-GDP ratios (%)

In [166]:
df["Debt_to_GDP"]= df["Debt_USD"]/ df["GDP_USD"] *100
print(df[["REF_AREA_LABEL", "TIME_PERIOD", "Debt_USD", "GDP_USD", "Debt_to_GDP"]].head())


  REF_AREA_LABEL  TIME_PERIOD  Debt_USD       GDP_USD  Debt_to_GDP
0      Argentina         1970       0.0  3.158421e+10          0.0
1      Argentina         1970       0.0  3.158421e+10          0.0
2      Argentina         1970       0.0  3.158421e+10          0.0
3      Argentina         1970       0.0  3.158421e+10          0.0
4      Argentina         1970       0.0  3.158421e+10          0.0


## Part 3: Visualization

### 1. Scatterplot for 2023

In [167]:
import plotly.express as px

last_year = df["TIME_PERIOD"].max()
sample = df[df["TIME_PERIOD"] == last_year]

print("Scatter year:", last_year, " sample size:", len(sample))

fig= px.scatter(
    sample,
    x="GDP_USD",
    y="Debt_USD",
    hover_name="REF_AREA_LABEL",
    title= f"Debt vs GDP (log-log), {last_year}",
    labels= {
        "GDP_USD": "GDP (current US$)",
        "Debt_USD": "External debt, long-term (current US$)"
    },
    log_x= True,
    log_y= True,
    opacity=0.6
)
fig.show()



Scatter year: 2023  sample size: 5386


### 2. Groupby Debt-to-GDP Ratio

In [176]:
yearly = df.groupby("TIME_PERIOD")["Debt_to_GDP"].mean().reset_index()

fig2= px.line(
    yearly,
    x="TIME_PERIOD",
    y="Debt_to_GDP",
    title="Average Debt-to-GDP Ratio Over Time",
    labels={
        "TIME_PERIOD": "Year",
        "Debt_to_GDP": "Average Debt-to-GDP Ratio (%)"
    }
)

fig2.update_layout(
    yaxis_tickformat=".1f",
    xaxis_title="Year",
    
)

fig2.show()

## Part 4: Conclusion

This analysis investigated the relationship between long-term external debt and economic size across countries from 2000–2023, using World Bank current-US$ data. The findings show a strong positive log-linear relationship between GDP and external debt: economies with larger output levels tend to borrow significantly more on international markets. This supports the hypothesis that economic scale expands access to external financing.

However, the results do not support the expectation that developing countries consistently face higher debt burdens relative to economic size. The global Debt-to-GDP ratio has declined over the past three decades, suggesting an improvement in debt sustainability worldwide. This pattern likely reflects structural changes, including reduced reliance on external financing, improved domestic capital markets, and regional or multilateral debt relief programs.

Despite this overall positive trend, some lower-income and fragile economies continue to exhibit elevated Debt-to-GDP ratios, indicating vulnerability to exchange-rate shocks and rising global interest rates. Policies should therefore remain focused on:

- maintaining fiscal discipline,
- ensuring access to concessional financing, and
- supporting timely debt restructuring where necessary.

Overall, the results highlight that economic growth has increasingly outpaced debt accumulation, contributing to more stable financial conditions globally—though challenges remain unevenly distributed across the international system.